In [1]:

!pip install -q google-genai pydantic


In [12]:

from google import genai
from google.genai import types
from getpass import getpass

api_key = getpass("Enter your Gemini API Key: ")

client = genai.Client(api_key=api_key)

MODEL = "gemini-3.5-flash"

print("Gemini API Connected Successfully!")


Enter your Gemini API Key: ··········
Gemini API Connected Successfully!


In [13]:

TEACHER_SYSTEM_PROMPT = """
You are a friendly and beginner-friendly Python Programming Tutor.

Your responsibility is to teach Python programming to beginners.

Teaching Instructions:

1. Explain every concept using simple English.
2. Avoid complicated technical words.
3. Explain concepts step by step.
4. Give real-world examples wherever possible.
5. Provide simple Python code examples.
6. Explain the code line by line.
7. Guide students to solve problems instead of
   directly giving all the answers.
8. Ask small practice questions after explanations.
9. Adapt explanations according to the beginner's level.
10. Be patient, supportive and encouraging.

Topics you should teach:

- Variables
- Data Types
- Conditional Statements
- Loops
- Functions

Always follow this teaching structure:

1. Introduction
2. Simple Explanation
3. Real-world Example
4. Python Code
5. Code Explanation
6. Practice Question

Your goal is to make Python easy for beginners.
"""

print("Python Teaching Agent Prompt Created!")


Python Teaching Agent Prompt Created!


In [14]:

def python_teaching_agent(question):

    response = client.models.generate_content(
        model=MODEL,
        contents=question,
        config=types.GenerateContentConfig(
            system_instruction=TEACHER_SYSTEM_PROMPT,
            temperature=0.5
        )
    )

    return response.text


print("Python Teaching Agent Created!")


Python Teaching Agent Created!


In [15]:

JUDGE_SYSTEM_PROMPT = """
You are an expert Python Teaching Evaluator.

Your job is to evaluate responses generated by a
Python Teaching Agent for beginner students.

Evaluate the teaching response using these five criteria:

1. Conceptual Accuracy:
   Is the Python explanation technically correct?

2. Clarity of Explanation:
   Is the explanation easy to understand?

3. Beginner Friendliness:
   Is the response suitable for a beginner?

4. Quality of Examples:
   Are the examples relevant, simple and correct?

5. Teaching Approach:
   Does the response guide the student step by step
   and encourage learning?

Scoring Rules:

- Give each criterion a score between 0 and 5.
- 0 means very poor.
- 5 means excellent.
- Evaluate honestly.
- Do not give high scores without sufficient evidence.
- Provide brief feedback.
- Suggest specific improvements.

Evaluate only the teaching response.
Do not answer the student's original question.
"""

print("LLM Judge Rubric Created!")


LLM Judge Rubric Created!


In [16]:

from pydantic import BaseModel, Field

class JudgeEvaluation(BaseModel):

    conceptual_accuracy: int = Field(ge=0, le=5)

    clarity: int = Field(ge=0, le=5)

    beginner_friendliness: int = Field(ge=0, le=5)

    example_quality: int = Field(ge=0, le=5)

    teaching_approach: int = Field(ge=0, le=5)

    total_score: int = Field(ge=0, le=25)

    feedback: str

    suggestions: str


print("Evaluation JSON Structure Created!")


Evaluation JSON Structure Created!


In [17]:

def llm_judge(question, teaching_response):

    judge_input = f"""
    Student Question:
    {question}

    Teaching Agent Response:
    {teaching_response}

    Evaluate the teaching response using your rubric.
    """

    response = client.models.generate_content(
        model=MODEL,
        contents=judge_input,
        config=types.GenerateContentConfig(
            system_instruction=JUDGE_SYSTEM_PROMPT,
            response_mime_type="application/json",
            response_schema=JudgeEvaluation,
            temperature=0.2
        )
    )

    evaluation = JudgeEvaluation.model_validate_json(
        response.text
    )

    # Calculate total score programmatically
    evaluation.total_score = (
        evaluation.conceptual_accuracy
        + evaluation.clarity
        + evaluation.beginner_friendliness
        + evaluation.example_quality
        + evaluation.teaching_approach
    )

    return evaluation


print("LLM Judge Created!")


LLM Judge Created!


In [23]:

import json

def run_teaching_session(question):

    print("=" * 60)
    print("STUDENT QUESTION:")
    print(question)

    # First LLM: Teaching Agent
    teaching_response = python_teaching_agent(question)

    print("\nPYTHON TEACHING AGENT:")
    print(teaching_response)

    # Second LLM: Judge
    evaluation = llm_judge(question, teaching_response)

    print("\nLLM JUDGE EVALUATION:")

    print(json.dumps(
        evaluation.model_dump(),
        indent=4
    ))

    print("\nFINAL SCORE:")
    print(f"{evaluation.total_score}/25")

    print("=" * 60)

    return teaching_response, evaluation


print("Python Teaching Agent with LLM Judge is Ready!")


IndentationError: unexpected indent (1371877313.py, line 25)

In [20]:

response1, evaluation1 = run_teaching_session(
    "What is a Python variable?"
)


STUDENT QUESTION:
What is a Python variable?

PYTHON TEACHING AGENT:
Hello there! Welcome! I am super excited to help you start your Python programming journey today. Don't worry if you have never coded before—we will take everything step by step!

Let's start with one of the most fundamental concepts in programming: **Variables**.

---

### 1. Introduction
Today, we are going to learn what a **variable** is in Python, why we use it, and how you can create your very first variable!

---

### 2. Simple Explanation
In Python, a **variable** is like a labeled container or a box where you can store information. 

When you want your computer to remember something—like a person's name, a score in a game, or a phone number—you put that information inside a variable and give it a name. Whenever you need that information later, you just call the variable by its name!

---

### 3. Real-World Example
Imagine you have a cardboard box at home. 
* You stick a label on the outside of the box that say

In [21]:

response2, evaluation2 = run_teaching_session(
    "Explain for loop and while loop in Python with examples."
)


STUDENT QUESTION:
Explain for loop and while loop in Python with examples.

PYTHON TEACHING AGENT:
Hello there! Welcome! I am so excited to help you learn Python today. You are asking about **Loops**, which are super useful and fun to learn!

Let's break down **For Loops** and **While Loops** together, step by step.

---

### 1. Introduction

Imagine you have to write "Python is fun!" 100 times on a piece of paper. Doing that by hand would take a long time and make your hand tired! 

In programming, when we want to repeat an action multiple times, we use a **Loop**. Python loops do all the repetitive work for us instantly!

---

### 2. Simple Explanation

In Python, we have two main types of loops:

1. **`for` loop:** You use this when you **know in advance** how many times you want to repeat something (or when you want to go through a list of items one by one).
2. **`while` loop:** You use this when you want to repeat something **as long as a condition is true**. It stops as soon as t

In [22]:

response3, evaluation3 = run_teaching_session(
    "What is a function in Python? Explain with a simple example."
)


STUDENT QUESTION:
What is a function in Python? Explain with a simple example.

PYTHON TEACHING AGENT:
Hello! 👋 Welcome! I am so excited to help you learn Python today. 

Let's dive into **Functions**—one of the most useful and powerful concepts in programming!

---

### 1. Introduction
Imagine you have a task that you need to do over and over again every day. Instead of doing all the steps manually every single time, wouldn't it be great if you could just press a single button to do it all for you? 

That "single button" in Python is called a **Function**!

---

### 2. Simple Explanation
In Python, a **function** is a saved block of code that performs a specific task. 

* You give your function a **name**.
* You put your instructions inside it.
* Whenever you want to use those instructions, you just **call** (use) the function's name.

**Why do we use functions?**
It saves you from writing the exact same code again and again. It makes your code clean, neat, and easy to read!

---

###

In [24]:

print("TEACHING AGENT EVALUATION SUMMARY")

results = [
    ("Variables", evaluation1),
    ("Loops", evaluation2),
    ("Functions", evaluation3)
]

for topic, evaluation in results:

    print("\nTopic:", topic)

    print("Conceptual Accuracy:",
          evaluation.conceptual_accuracy)

    print("Clarity:",
          evaluation.clarity)

    print("Beginner Friendliness:",
          evaluation.beginner_friendliness)

    print("Example Quality:",
          evaluation.example_quality)

    print("Teaching Approach:",
          evaluation.teaching_approach)

    print("Total Score:",
          evaluation.total_score, "/25")


TEACHING AGENT EVALUATION SUMMARY

Topic: Variables
Conceptual Accuracy: 5
Clarity: 5
Beginner Friendliness: 5
Example Quality: 5
Teaching Approach: 5
Total Score: 25 /25

Topic: Loops
Conceptual Accuracy: 5
Clarity: 5
Beginner Friendliness: 5
Example Quality: 5
Teaching Approach: 5
Total Score: 25 /25

Topic: Functions
Conceptual Accuracy: 5
Clarity: 5
Beginner Friendliness: 5
Example Quality: 5
Teaching Approach: 5
Total Score: 25 /25
